# ScentTree Taxonomy Analysis
**Mapping GoodScents/Leffingwell Fine Labels to ScentTree Layer 1 Nodes**

This notebook documents the full analysis performed to:
1. Understand the structure of the ScentTree JSON taxonomy
2. Match the 138 fine-grained odor descriptors from the dataset to ScentTree nodes
3. Compare two mapping strategies: **all-paths** vs. **2-hop**
4. Propose Layer 1 parent assignments for the 75 labels absent from ScentTree

## 0. Imports and Setup

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 150)

SCENTREE_PATH = 'scent_tree_cleaned.json'  # update path if needed

## 1. Load and Explore the ScentTree JSON

ScentTree is a hierarchical olfactory taxonomy provided by the project supervisor.
Each node has a `name` and a list of `children`.

In [ ]:
with open(SCENTREE_PATH) as f:
    tree = json.load(f)

print(f"Number of Layer 1 (root) nodes: {len(tree)}")
print("\nLayer 1 nodes:")
for node in tree:
    print(f"  {node['name']}  ({len(node['children'])} direct children)")

### 1.1 Count all unique node names across all depths

In [ ]:
def collect_all_nodes(nodes, depth=0, parent=None, records=None):
    """Recursively collect all nodes with their depth and parent."""
    if records is None:
        records = []
    for node in nodes:
        records.append({'name': node['name'], 'depth': depth, 'parent': parent})
        collect_all_nodes(node['children'], depth + 1, node['name'], records)
    return records

all_records = collect_all_nodes(tree)
all_unique_names = set(r['name'].lower() for r in all_records)

print(f"Total node occurrences (with duplicates across branches): {len(all_records)}")
print(f"Total UNIQUE node names across all depths:                {len(all_unique_names)}")

# Distribution by depth
depth_counts = defaultdict(int)
for r in all_records:
    depth_counts[r['depth']] += 1

print("\nNode occurrences by depth (nodes appear multiple times due to DAG):")
for d in sorted(depth_counts):
    label = 'Layer 1 (roots)' if d == 0 else f'Layer {d+1}'
    print(f"  Depth {d} ({label}): {depth_counts[d]} occurrences")

### 1.2 Understand the DAG structure

A key property of ScentTree is that it is a **DAG (Directed Acyclic Graph)**, not a strict tree.
This means a node can have **multiple parents** — a single olfactory concept can belong to several families.

This is intentional: it reflects the genuine multi-faceted nature of odors.

In [ ]:
# Build: for each Layer 1 node, get all descendants with their shallowest depth
def get_descendants_with_depth(node, current_depth=0):
    """Return list of (name, shallowest_depth) for all descendants."""
    result = []
    for child in node['children']:
        result.append((child['name'], current_depth + 1))
        result.extend(get_descendants_with_depth(child, current_depth + 1))
    return result

layer1_descendants = {}
for node in tree:
    raw = get_descendants_with_depth(node)
    # Keep only the shallowest depth for each unique descendant name
    best = {}
    for name, depth in raw:
        if name not in best or depth < best[name]:
            best[name] = depth
    layer1_descendants[node['name']] = best  # {node_name: shallowest_depth}

# For each unique node name, count how many Layer 1 roots it appears under
node_parent_count = defaultdict(list)
for l1_name, descendants in layer1_descendants.items():
    for node_name in descendants:
        node_parent_count[node_name].append(l1_name)

multi_parent = {k: v for k, v in node_parent_count.items() if len(v) > 1}
print(f"Nodes appearing under multiple Layer 1 parents: {len(multi_parent)} / {len(node_parent_count)}")
print(f"\nTop 10 most 'shared' nodes:")
top_shared = sorted(multi_parent.items(), key=lambda x: -len(x[1]))[:10]
for name, parents in top_shared:
    print(f"  '{name}' -> {len(parents)} parents: {parents}")

## 2. Define the Dataset Fine Labels

In [ ]:
DATASET_LABELS = [
    'alcoholic','aldehydic','alliaceous','almond','amber','animal','anisic','apple','apricot',
    'aromatic','balsamic','banana','beefy','bergamot','berry','bitter','black currant','brandy',
    'burnt','buttery','cabbage','camphoreous','caramellic','cedar','celery','chamomile','cheesy',
    'cherry','chocolate','cinnamon','citrus','clean','clove','cocoa','coconut','coffee','cognac',
    'cooked','cooling','cortex','coumarinic','creamy','cucumber','dairy','dry','earthy','ethereal',
    'fatty','fermented','fishy','floral','fresh','fruit skin','fruity','garlic','gassy','geranium',
    'grape','grapefruit','grassy','green','hawthorn','hay','hazelnut','herbal','honey','hyacinth',
    'jasmin','juicy','ketonic','lactonic','lavender','leafy','leathery','lemon','lily','malty',
    'meaty','medicinal','melon','metallic','milky','mint','muguet','mushroom','musk','musty',
    'natural','nutty','odorless','oily','onion','orange','orangeflower','orris','ozone','peach',
    'pear','phenolic','pine','pineapple','plum','popcorn','potato','powdery','pungent','radish',
    'raspberry','ripe','roasted','rose','rummy','sandalwood','savory','sharp','smoky','soapy',
    'solvent','sour','spicy','strawberry','sulfurous','sweaty','sweet','tea','terpenic','tobacco',
    'tomato','tropical','vanilla','vegetable','vetiver','violet','warm','waxy','weedy','winey','woody'
]

print(f"Total fine-grained labels in dataset: {len(DATASET_LABELS)}")

## 3. Match Dataset Labels to ScentTree Nodes

We use two methods:
- **Exact match**: label name == ScentTree node name (case-insensitive)
- **Manual synonym mapping**: for labels with slightly different names (e.g. 'jasmin' → 'Jasmine', 'rose' → 'Rosy')

In [ ]:
# Manual synonym table: dataset label -> ScentTree node name
MANUAL_MAP = {
    'aldehydic':   'Aldehydes',
    'almond':      'Almondy',
    'amber':       'Balsamic Ambery',
    'animal':      'Animalic',
    'berry':       'Berries',
    'camphoreous': 'Camphoric',
    'cinnamon':    'Cinnamic',
    'citrus':      'Citrus',
    'floral':      'Floral',
    'fresh':       'Fresh Flowers',
    'fruity':      'Fruity',
    'green':       'Green',
    'herbal':      'Herbal',
    'honey':       'Honeyed',
    'jasmin':      'Jasmine',
    'juicy':       'Juicy Fruits',
    'leathery':    'Leather',
    'lemon':       'Citric',
    'mint':        'Minty',
    'musk':        'Musky',
    'musty':       'Mossy',
    'orangeflower':'Orange Blossom',
    'orris':       'Orris Root',
    'ozone':       'Ozonic',
    'pine':        'Coniferous',
    'rose':        'Rosy',
    'smoky':       'Smoky Woods',
    'solvent':     'Solvents',
    'spicy':       'Spicy',
    'sulfurous':   'Sulfuric',
    'tropical':    'Tropical Fruits',
    'vanilla':     'Vanillic',
    'violet':      'Violet Flower',
    'woody':       'Woody',
}

def get_scentree_name(label):
    """Return the matching ScentTree node name for a dataset label, or None."""
    if label in MANUAL_MAP:
        return MANUAL_MAP[label], 'manual'
    if label.lower() in all_unique_names:
        # find the original capitalisation
        for r in all_records:
            if r['name'].lower() == label.lower():
                return r['name'], 'exact'
    return None, None

# Classify all labels
match_results = []
for label in DATASET_LABELS:
    sname, method = get_scentree_name(label)
    match_results.append({'label': label, 'scentree_node': sname, 'match_method': method})

df_match = pd.DataFrame(match_results)

print("=== MATCHING SUMMARY ===")
print(df_match['match_method'].value_counts(dropna=False).rename({None: 'not in ScentTree'}))
print(f"\nTotal matched:       {df_match['scentree_node'].notna().sum()}")
print(f"Total NOT in tree:   {df_match['scentree_node'].isna().sum()}")

In [ ]:
# Show the 63 matched labels
matched = df_match[df_match['scentree_node'].notna()].copy()
print("Labels matched to ScentTree nodes:")
matched[['label','scentree_node','match_method']]

In [ ]:
# Show the 75 unmatched labels
unmatched = df_match[df_match['scentree_node'].isna()]['label'].tolist()
print(f"Labels ABSENT from ScentTree ({len(unmatched)}):")
print(unmatched)

### Why are 75 labels absent?

This is **not** because the JSON is incomplete. After checking all 101 unique node names across all depths, these labels are confirmed to be genuinely absent.

The reason is a **deliberate design difference**:
- Our dataset uses **specific, concrete descriptors**: `apple`, `strawberry`, `garlic`, `chocolate`
- ScentTree uses **abstract family-level names**: `Yellow Fruits`, `Berries`, `Sulfuric`, `Gourmand`

ScentTree operates at a higher level of abstraction by design. Mapping these 75 labels is a legitimate semantic bridging task, not a workaround.

## 4. Compute Layer 1 Parent Mappings

For each matched label, we find which Layer 1 root nodes it appears under in ScentTree.
We compare two strategies:
- **All-paths**: count a Layer 1 parent regardless of how deep the connection is
- **2-hop**: only count a Layer 1 parent if the label appears within depth 1 or 2 of that root

In [ ]:
layer1_names = [node['name'] for node in tree]

def find_layer1_parents(scentree_name, max_depth=None):
    """
    Return list of Layer 1 nodes that contain scentree_name as a descendant.
    max_depth: if set, only count connections at shallowest_depth <= max_depth.
               If None, include all depths (all-paths strategy).
    Also handles the case where the name IS a Layer 1 node itself (depth 0).
    """
    parents = []
    sname_lower = scentree_name.lower()

    for l1_name, descendants in layer1_descendants.items():
        for desc_name, depth in descendants.items():
            if desc_name.lower() == sname_lower:
                if max_depth is None or depth <= max_depth:
                    parents.append(l1_name)
                break

    # If the node itself is a Layer 1 root
    for l1_node in tree:
        if l1_node['name'].lower() == sname_lower and l1_node['name'] not in parents:
            parents.append(l1_node['name'])

    return sorted(set(parents))

# Build mappings for all matched labels
all_paths_map = {}
two_hop_map   = {}

for _, row in matched.iterrows():
    label  = row['label']
    sname  = row['scentree_node']
    all_paths_map[label] = find_layer1_parents(sname, max_depth=None)
    two_hop_map[label]   = find_layer1_parents(sname, max_depth=2)

print("Example: 'rose'")
print(f"  All-paths: {all_paths_map['rose']}")
print(f"  2-hop:     {two_hop_map['rose']}")
print()
print("Example: 'camphoreous'")
print(f"  All-paths: {all_paths_map['camphoreous']}")
print(f"  2-hop:     {two_hop_map['camphoreous']}")

## 5. Strategy Comparison: All-paths vs. 2-hop

The 2-hop rule filters out distant, weak associations (e.g. 'Rosy' appearing under Marine at depth 3 because some marine compounds have a rosy facet), keeping only primary, well-established connections.

In [ ]:
all_counts = [len(v) for v in all_paths_map.values()]
two_counts = [len(v) for v in two_hop_map.values()]

stats = {
    'Metric': [
        'Average metacategories per fine label',
        'Maximum metacategories (single label)',
        'Minimum metacategories',
        'Labels with exactly 1 parent',
        'Labels with exactly 2 parents',
        'Labels with exactly 3 parents',
        'Labels with 4+ parents',
        'Labels with 0 parents (lost all connections)',
    ],
    'All-paths': [
        round(sum(all_counts) / len(all_counts), 2),
        max(all_counts),
        min(all_counts),
        sum(c == 1 for c in all_counts),
        sum(c == 2 for c in all_counts),
        sum(c == 3 for c in all_counts),
        sum(c >= 4 for c in all_counts),
        sum(c == 0 for c in all_counts),
    ],
    '2-hop': [
        round(sum(two_counts) / len(two_counts), 2),
        max(two_counts),
        min(two_counts),
        sum(c == 1 for c in two_counts),
        sum(c == 2 for c in two_counts),
        sum(c == 3 for c in two_counts),
        sum(c >= 4 for c in two_counts),
        sum(c == 0 for c in two_counts),
    ]
}

df_stats = pd.DataFrame(stats).set_index('Metric')
print(df_stats.to_string())

### 5.1 Why do labels with 1 parent *increase* under 2-hop?

The 2-hop rule only removes connections — it never adds them. Labels that had 2–3 parents via distant (depth 3+) connections lose those parents and drop to 1. Example:
- `camphoreous` (→ Camphoric): all-paths gives `[Herbal, Undergrowth, Woody]`; 2-hop gives `[Herbal]` because Camphoric only appears within 2 hops of Herbal.

In [ ]:
# Visualise the distribution of parent counts
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

max_count = max(max(all_counts), max(two_counts))
bins = range(0, max_count + 2)

for ax, counts, title, color in zip(
    axes,
    [all_counts, two_counts],
    ['All-paths strategy', '2-hop strategy (recommended)'],
    ['#4472C4', '#70AD47']
):
    ax.hist(counts, bins=bins, align='left', color=color, edgecolor='white', rwidth=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Number of Layer 1 parents', fontsize=11)
    ax.set_ylabel('Number of fine labels', fontsize=11)
    ax.axvline(sum(counts)/len(counts), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean = {sum(counts)/len(counts):.2f}')
    ax.legend(fontsize=10)
    ax.set_xticks(list(bins)[:-1])

plt.suptitle('Distribution of Layer 1 parent count per fine label\n(63 labels matched to ScentTree)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('parent_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as parent_distribution.png")

In [ ]:
# Full per-label comparison table
rows = []
for label in sorted(all_paths_map):
    ap = all_paths_map[label]
    th = two_hop_map[label]
    lost = [p for p in ap if p not in th]
    rows.append({
        'fine label':        label,
        'scentree node':     matched.set_index('label').loc[label, 'scentree_node'],
        'all-paths parents': ', '.join(ap),
        '# all':             len(ap),
        '2-hop parents':     ', '.join(th),
        '# 2-hop':           len(th),
        'connections lost':  ', '.join(lost) if lost else '—',
    })

df_comparison = pd.DataFrame(rows)
df_comparison

## 6. Proposed Mapping for the 75 Absent Labels

For each label absent from ScentTree, we identify the closest ScentTree node(s) by semantic reasoning
and propose Layer 1 parent(s). Confidence reflects how clear the assignment is:
- **Strong**: unambiguous, well-established olfactory connection
- **Medium**: reasonable but slightly debatable
- **Weak**: best guess; needs expert review
- **Ambiguous**: not a genuine odor category — should be discussed with supervisor

In [ ]:
UNMATCHED_PROPOSALS = [
    # (fine_label, closest_scentree_node, proposed_layer1_parents, confidence)
    # --- Fruity ---
    ('apple',        'Yellow Fruits / Juicy Fruits',   ['Fruity'],                              'Strong'),
    ('apricot',      'Yellow Fruits',                   ['Fruity'],                              'Strong'),
    ('banana',       'Yellow Fruits / Tropical Fruits', ['Fruity'],                              'Strong'),
    ('black currant','Berries',                          ['Fruity', 'Sulfuric'],                 'Strong'),
    ('cherry',       'Berries / Juicy Fruits',           ['Fruity'],                              'Strong'),
    ('grape',        'Berries / Juicy Fruits',           ['Fruity'],                              'Strong'),
    ('melon',        'Juicy Fruits / Green Fruits',      ['Fruity', 'Green'],                    'Strong'),
    ('peach',        'Yellow Fruits / Juicy Fruits',     ['Fruity'],                              'Strong'),
    ('pear',         'Green Fruits / Yellow Fruits',     ['Fruity'],                              'Strong'),
    ('pineapple',    'Tropical Fruits',                  ['Fruity', 'Sulfuric'],                 'Strong'),
    ('plum',         'Yellow Fruits / Berries',          ['Fruity'],                              'Strong'),
    ('raspberry',    'Berries',                          ['Fruity', 'Sulfuric'],                 'Strong'),
    ('strawberry',   'Berries / Juicy Fruits',           ['Fruity'],                              'Strong'),
    ('ripe',         'Juicy Fruits / Yellow Fruits',     ['Fruity'],                              'Medium'),
    ('juicy',        'Juicy Fruits',                     ['Fruity', 'Green'],                    'Strong'),
    ('fruit skin',   'Green Fruits / Zesty',             ['Fruity', 'Citrus'],                   'Medium'),
    # --- Gourmand / Sweet ---
    ('caramellic',   'Gourmand',                         ['Balsamic Ambery'],                    'Strong'),
    ('chocolate',    'Gourmand / Roasted',               ['Balsamic Ambery', 'Burnt Leather'],  'Strong'),
    ('cocoa',        'Gourmand / Roasted',               ['Balsamic Ambery', 'Burnt Leather'],  'Strong'),
    ('coffee',       'Roasted',                          ['Balsamic Ambery', 'Burnt Leather'],  'Strong'),
    ('sweet',        'Gourmand / Vanillic',              ['Balsamic Ambery'],                    'Medium'),
    ('creamy',       'Lactonic / Buttery',               ['Fruity', 'Woody'],                   'Medium'),
    ('dairy',        'Lactonic / Butyric',               ['Butyric Buttery', 'Animalic'],       'Medium'),
    ('popcorn',      'Roasted / Gourmand',               ['Balsamic Ambery', 'Burnt Leather'],  'Medium'),
    ('hazelnut',     'Nutty',                            ['Balsamic Ambery', 'Fruity', 'Woody'],'Strong'),
    # --- Animal / Savory ---
    ('beefy',        'Feacal / Butyric',                 ['Animalic', 'Butyric Buttery'],       'Medium'),
    ('meaty',        'Feacal / Butyric',                 ['Animalic', 'Burnt Leather'],         'Medium'),
    ('cheesy',       'Butyric',                          ['Butyric Buttery', 'Animalic'],       'Strong'),
    ('fishy',        'Feacal / Animalic',                ['Animalic', 'Marine'],                'Medium'),
    ('sweaty',       'Feacal / Butyric',                 ['Animalic', 'Butyric Buttery'],       'Medium'),
    ('savory',       'Eugenol / Spicy',                  ['Spicy', 'Herbal'],                   'Medium'),
    ('cooked',       'Roasted / Gourmand',               ['Burnt Leather', 'Balsamic Ambery'],  'Weak'),
    # --- Green / Vegetal ---
    ('cucumber',     'Aquatic / Crisp Green',            ['Green', 'Marine'],                   'Medium'),
    ('tomato',       'Crisp Green / Agrestic',           ['Green', 'Herbal'],                   'Medium'),
    ('vegetable',    'Agrestic / Crisp Green',           ['Green', 'Undergrowth'],               'Medium'),
    ('leafy',        'Cut Grass / Crisp Green',          ['Green'],                              'Strong'),
    ('hay',          'Cut Grass / Agrestic',             ['Green', 'Herbal', 'Undergrowth'],    'Strong'),
    ('weedy',        'Agrestic / Cut Grass',             ['Green', 'Undergrowth'],               'Strong'),
    ('celery',       'Agrestic / Green',                 ['Green', 'Herbal'],                   'Medium'),
    ('cabbage',      'Agrestic / Sulfuric',              ['Green', 'Sulfuric'],                 'Medium'),
    ('garlic',       'Sulfuric',                         ['Sulfuric', 'Green'],                 'Strong'),
    ('onion',        'Sulfuric',                         ['Sulfuric', 'Green'],                 'Strong'),
    ('radish',       'Sulfuric / Agrestic',              ['Sulfuric', 'Green'],                 'Medium'),
    ('potato',       'Agrestic / Earthy',                ['Undergrowth', 'Green'],              'Medium'),
    # --- Chemical / Solvent ---
    ('ethereal',     'Etheric Solvent',                  ['Solvents', 'Fruity'],                'Strong'),
    ('gassy',        'Etheric Solvent / Sulfuric',       ['Solvents', 'Sulfuric'],              'Medium'),
    ('pungent',      'Sulfuric / Etheric Solvent',       ['Sulfuric', 'Solvents'],              'Medium'),
    ('phenolic',     'Eugenol / Etheric Solvent',        ['Spicy', 'Burnt Leather'],            'Medium'),
    ('soapy',        'Aldehydes / Light Flowers',        ['Aldehydes', 'Floral'],               'Medium'),
    ('sharp',        'Etheric Solvent / Sulfuric',       ['Solvents', 'Sulfuric'],              'Weak'),
    ('clean',        'Ozonic / Aquatic',                 ['Marine', 'Aldehydes'],               'Weak'),
    ('dry',          'Dry Woods / Agrestic',             ['Woody', 'Undergrowth'],              'Weak'),
    ('ketonic',      'Etheric Solvent',                  ['Solvents'],                          'Medium'),
    # --- Spicy / Herbal ---
    ('clove',        'Eugenol / Warm Spices',            ['Spicy', 'Balsamic Ambery'],          'Strong'),
    ('chamomile',    'Herbal / Fresh Flowers',           ['Herbal', 'Floral'],                  'Strong'),
    ('hawthorn',     'White Flowers / Fresh Flowers',    ['Floral', 'Herbal'],                  'Medium'),
    ('cooling',      'Icy / Cool Spices / Minty',        ['Herbal', 'Spicy'],                   'Medium'),
    ('cortex',       'Cinnamic / Warm Spices',           ['Spicy', 'Balsamic Ambery'],          'Medium'),
    # --- Floral ---
    ('hyacinth',     'Fresh Flowers / White Flowers',    ['Floral'],                             'Strong'),
    ('lily',         'White Flowers / Light Flowers',    ['Floral'],                             'Strong'),
    ('muguet',       'Fresh Flowers / Light Flowers',    ['Floral', 'Green'],                   'Strong'),
    # --- Boozy ---
    ('alcoholic',    'Boozy',                            ['Fruity', 'Balsamic Ambery'],         'Strong'),
    ('brandy',       'Boozy',                            ['Balsamic Ambery', 'Fruity'],         'Strong'),
    ('cognac',       'Boozy',                            ['Balsamic Ambery', 'Fruity'],         'Strong'),
    ('rummy',        'Boozy',                            ['Balsamic Ambery', 'Fruity'],         'Strong'),
    ('winey',        'Boozy / Berries',                  ['Fruity', 'Balsamic Ambery'],         'Strong'),
    ('fermented',    'Boozy / Butyric',                  ['Animalic', 'Balsamic Ambery'],       'Medium'),
    ('malty',        'Boozy / Roasted',                  ['Balsamic Ambery', 'Burnt Leather'], 'Medium'),
    # --- Modifiers / Ambiguous ---
    ('warm',         'Warm Spices / Warm Woods',         ['Spicy', 'Woody', 'Balsamic Ambery'],'Weak'),
    ('aromatic',     'Herbal / Terpenic',                ['Herbal', 'Spicy'],                   'Weak'),
    ('bitter',       'Eugenol / Citric',                 ['Citrus', 'Herbal'],                  'Weak'),
    ('natural',      '—',                               [],                                     'Ambiguous'),
    ('alliaceous',   'Sulfuric',                         ['Sulfuric'],                           'Strong'),
    ('oily',         'Fatty / Waxy',                     ['Green', 'Woody'],                    'Weak'),
    ('odorless',     '—',                               [],                                     'Ambiguous'),
    ('sour',         'Citric / Zesty',                   ['Citrus', 'Sulfuric'],                'Weak'),
]

df_unmatched = pd.DataFrame(UNMATCHED_PROPOSALS,
    columns=['fine label', 'closest ScentTree node', 'proposed Layer 1 parents', 'confidence'])
df_unmatched['proposed Layer 1 parents'] = df_unmatched['proposed Layer 1 parents'].apply(
    lambda x: ', '.join(x) if x else 'Drop / discuss')

print(f"Total proposals: {len(df_unmatched)}")
print("\nConfidence breakdown:")
print(df_unmatched['confidence'].value_counts())
df_unmatched

In [ ]:
# Confidence breakdown bar chart
conf_order  = ['Strong', 'Medium', 'Weak', 'Ambiguous']
conf_colors = ['#70AD47', '#4472C4', '#FFC000', '#FF0000']
conf_counts = df_unmatched['confidence'].value_counts().reindex(conf_order, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(conf_order, conf_counts.values, color=conf_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, conf_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('Confidence of proposed Layer 1 assignments\n(75 labels absent from ScentTree)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Number of fine labels', fontsize=11)
ax.set_ylim(0, max(conf_counts.values) + 4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Build the Final META_CATEGORIES Dictionary

Combining the 63 matched labels (using the **2-hop strategy**) and the 75 manually assigned labels,
we construct the new `META_CATEGORIES` dictionary in the same format as Vassilis's original.

In [ ]:
META_CATEGORIES = defaultdict(list)

# 1. Add the 63 matched labels using the 2-hop strategy
for label, parents in two_hop_map.items():
    for parent in parents:
        META_CATEGORIES[parent].append(label)

# 2. Add the 75 manually assigned labels
for _, row in df_unmatched.iterrows():
    label    = row['fine label']
    parents  = row['proposed Layer 1 parents']
    confidence = row['confidence']
    if confidence == 'Ambiguous':
        continue  # skip 'natural' and 'odorless'
    for parent in [p.strip() for p in parents.split(',') if p.strip() and p.strip() != 'Drop / discuss']:
        META_CATEGORIES[parent].append(label)

# Sort for readability
META_CATEGORIES = {k: sorted(set(v)) for k, v in sorted(META_CATEGORIES.items())}

print("=== NEW META_CATEGORIES DICTIONARY (ScentTree-based) ===")
print(f"Number of metacategories: {len(META_CATEGORIES)}\n")
for cat, labels in META_CATEGORIES.items():
    print(f"  '{cat}' ({len(labels)} labels):")
    print(f"    {labels}")
    print()

In [ ]:
# Coverage check: how many fine labels are covered?
all_covered = set(label for labels in META_CATEGORIES.values() for label in labels)
not_covered = set(DATASET_LABELS) - all_covered

print(f"Fine labels covered by new dictionary: {len(all_covered)} / {len(DATASET_LABELS)}")
print(f"Fine labels NOT covered (ambiguous/dropped): {sorted(not_covered)}")

In [ ]:
# Size comparison: old (12 categories) vs new (17 categories)
OLD_META_CATEGORIES = {
    'floral':        ['floral','rose','jasmin','lily','violet','hyacinth','lavender','geranium',
                      'muguet','orangeflower','hawthorn','orris','chamomile','fresh'],
    'fruity':        ['fruity','apple','peach','apricot','pear','cherry','berry','grape','melon',
                      'pineapple','banana','plum','raspberry','strawberry','black currant',
                      'bergamot','lemon','orange','grapefruit','tropical','coconut','lactonic',
                      'ripe','fruit skin','juicy'],
    'sweet':         ['sweet','vanilla','caramellic','honey','buttery','creamy','waxy','powdery'],
    'woody':         ['woody','cedar','sandalwood','vetiver','smoky','roasted','burnt','tobacco'],
    'green':         ['green','grassy','hay','leafy','weedy','pine','herbal','tea','terpenic',
                      'camphoreous','mint','cooling','fresh'],
    'spicy':         ['spicy','cinnamon','clove','anisic','cortex','savory','balsamic','coumarinic'],
    'animal_musk':   ['animal','musk','musty','leathery','dairy','cheesy','beefy','meaty',
                      'sweaty','fishy','fermented'],
    'earthy':        ['earthy','mushroom','potato','celery','tomato','vegetable','mushroom'],
    'citrus':        ['citrus','lemon','orange','grapefruit','bergamot','ozone'],
    'chemical':      ['chemical','solvent','ethereal','aldehydic','phenolic','metallic','medicinal',
                      'sulfurous','alliaceous','garlic','onion','pungent','sharp','soapy'],
    'gourmand':      ['gourmand','chocolate','cocoa','coffee','hazelnut','nutty','caramellic',
                      'malty','brandy','cognac','rummy','winey','alcoholic','popcorn'],
    'powdery_amber': ['powdery','amber','balsamic','vanilla','orris','violet','iris'],
}

print("=== SIZE COMPARISON ===")
print(f"Old dictionary: {len(OLD_META_CATEGORIES)} metacategories")
print(f"New dictionary: {len(META_CATEGORIES)} metacategories")

print("\nOld category sizes:")
for cat, labels in sorted(OLD_META_CATEGORIES.items(), key=lambda x: -len(x[1])):
    print(f"  {cat:<20}: {len(labels)} labels")

print("\nNew category sizes (ScentTree-based):")
for cat, labels in sorted(META_CATEGORIES.items(), key=lambda x: -len(x[1])):
    print(f"  {cat:<25}: {len(labels)} labels")

In [ ]:
# Visualise category sizes side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Old
old_cats   = list(OLD_META_CATEGORIES.keys())
old_sizes  = [len(OLD_META_CATEGORIES[c]) for c in old_cats]
axes[0].barh(old_cats, old_sizes, color='#4472C4', edgecolor='white')
axes[0].set_title('Original dictionary\n(12 metacategories)', fontweight='bold')
axes[0].set_xlabel('Number of fine labels')
axes[0].invert_yaxis()
for i, v in enumerate(old_sizes):
    axes[0].text(v + 0.1, i, str(v), va='center', fontsize=9)

# New
new_cats  = list(META_CATEGORIES.keys())
new_sizes = [len(META_CATEGORIES[c]) for c in new_cats]
sorted_new = sorted(zip(new_cats, new_sizes), key=lambda x: -x[1])
new_cats_s, new_sizes_s = zip(*sorted_new)
axes[1].barh(new_cats_s, new_sizes_s, color='#70AD47', edgecolor='white')
axes[1].set_title('ScentTree-based dictionary\n(17 metacategories, 2-hop)', fontweight='bold')
axes[1].set_xlabel('Number of fine labels')
axes[1].invert_yaxis()
for i, v in enumerate(new_sizes_s):
    axes[1].text(v + 0.1, i, str(v), va='center', fontsize=9)

plt.suptitle('Metacategory size comparison: original vs. ScentTree-based dictionary',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('category_size_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Points for Supervisor Discussion

| Topic | Recommendation |
|---|---|
| Depth strategy | **2-hop recommended** — filters weak distant associations, avg 2.86 parents/label vs 3.83 |
| Modifier labels | `warm`, `aromatic`, `clean`, `dry`, `sharp`, `pungent`, `bitter`, `sour`, `ripe` — describe *how*, not *what* |
| Labels to drop | `odorless`, `natural` — not olfactory categories, cannot be mapped |
| Butyric Buttery | ScentTree's dedicated node captures `cheesy`, `dairy`, `sweaty` more accurately than `animal_musk` |
| Sulfuric coverage | `garlic`, `onion`, `alliaceous`, `cabbage`, `radish`, `black currant` correctly land in Sulfuric |
| 17 vs 12 categories | Some new nodes sparse (Butyric Buttery, Aldehydes, Solvents) — discuss merging if needed |

---
*Note: the 2-hop assignment drops 2 labels (`powdery`, `waxy`) to 0 parents — these need manual assignment.*